In [2]:
!pip install -q google-generativeai

In [ ]:
import pandas as pd
import requests
import time

# =========================
# CONFIG
# =========================

SERPER_API_KEY=your_key_here

# =========================
# LOAD DATA
# =========================

# Excel file should contain:
# Company Name

df = pd.read_excel("companies.xlsx")

# =========================
# SECTOR SCORE MAP
# =========================

sector_score_map = {
    "Electronic chemicals": 20,
    "Specialty chemicals": 20,
    "Pharma intermediates": 20,
    "Agrochemical intermediates": 18,
    "Polymer additives": 16,
    "Water treatment chemicals": 15,
    "Commodity/basic chemicals": 8,
    "CRO/service/testing": 2,
    "Other": 5
}

# =========================
# SECTOR REASON MAP
# =========================

sector_reason_map = {

    "Electronic chemicals":
        "Benefits from semiconductor manufacturing expansion, China+1 tailwinds, and Make-in-India initiatives.",

    "Specialty chemicals":
        "Benefits from export-oriented specialty manufacturing, differentiated chemistry, and China+1 tailwinds.",

    "Pharma intermediates":
        "Benefits from API localization, pharmaceutical exports, and supply-chain diversification away from China.",

    "Agrochemical intermediates":
        "Supported by agrochemical export growth and global supplier diversification.",

    "Polymer additives":
        "Benefits from industrial materials demand and downstream manufacturing growth.",

    "Water treatment chemicals":
        "Driven by sustainability initiatives and infrastructure investments.",

    "Commodity/basic chemicals":
        "Lower differentiation and relatively weaker structural growth tailwinds.",

    "CRO/service/testing":
        "Service-oriented business with lower manufacturing-led growth tailwinds.",

    "Other":
        "Insufficient evidence for strong sector tailwinds or differentiated specialty manufacturing."
}

# =========================
# SERPER SEARCH FUNCTION
# =========================

def serper_search(company_name):

    url = "https://google.serper.dev/search"

    payload = {
        "q": f"{company_name} specialty chemicals pharma intermediates manufacturing products"
    }

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:

        response = requests.post(
            url,
            headers=headers,
            json=payload
        )

        if response.status_code != 200:
            print(f"Serper Error {response.status_code} for {company_name}")
            return ""

        data = response.json()

        snippets = []

        if "organic" in data:

            for item in data["organic"][:5]:

                title = item.get("title", "")
                snippet = item.get("snippet", "")
                link = item.get("link", "")

                snippets.append(
                    f"TITLE: {title}\n"
                    f"SNIPPET: {snippet}\n"
                    f"LINK: {link}"
                )

        return "\n\n".join(snippets)

    except Exception as e:

        print(f"Search Error for {company_name}: {e}")
        return ""

# =========================
# SECTOR CLASSIFICATION
# =========================

def classify_sector(text):

    text = text.lower()

    # -------------------------
    # CRO / SERVICES
    # -------------------------

    if any(k in text for k in [
        "cro",
        "crdm",
        "cdmo",
        "contract research",
        "testing services"
    ]):
        return "CRO/service/testing"

    # -------------------------
    # ELECTRONIC CHEMICALS
    # -------------------------

    elif any(k in text for k in [
        "electronic chemicals",
        "semiconductor",
        "display chemicals"
    ]):
        return "Electronic chemicals"

    # -------------------------
    # AGROCHEM
    # -------------------------

    elif any(k in text for k in [
        "agrochemical",
        "crop protection",
        "pesticide intermediate"
    ]):
        return "Agrochemical intermediates"

    # -------------------------
    # POLYMERS / RESINS
    # -------------------------

    elif any(k in text for k in [
        "resin",
        "resins",
        "polymer",
        "polymers",
        "coatings",
        "rubber products",
        "gaskets"
    ]):
        return "Polymer additives"

    # -------------------------
    # SPECIALTY CHEMICALS
    # -------------------------

    elif any(k in text for k in [
        "specialty chemicals",
        "speciality chemicals",
        "fine chemicals",
        "custom synthesis",
        "organometallic",
        "niche intermediates",
        "advanced intermediates"
    ]):
        return "Specialty chemicals"

    # -------------------------
    # PHARMA INTERMEDIATES
    # -------------------------

    elif any(k in text for k in [
        "api",
        "active pharmaceutical ingredient",
        "pharmaceutical intermediates",
        "drug intermediates",
        "bulk drugs"
    ]):
        return "Pharma intermediates"

    # -------------------------
    # WATER TREATMENT
    # -------------------------

    elif any(k in text for k in [
        "water treatment"
    ]):
        return "Water treatment chemicals"

    # -------------------------
    # FALLBACK
    # -------------------------

    else:
        return "Other"

# =========================
# CONFIDENCE FUNCTION
# =========================

def get_confidence(sector, text):

    if sector == "Other":
        return "Low"

    keyword_counts = {

        "Specialty chemicals": [
            "specialty chemicals",
            "fine chemicals",
            "custom synthesis",
            "advanced intermediates"
        ],

        "Pharma intermediates": [
            "api",
            "pharmaceutical intermediates",
            "bulk drugs"
        ],

        "Polymer additives": [
            "resin",
            "polymer",
            "coatings"
        ]
    }

    text = text.lower()

    matches = 0

    if sector in keyword_counts:

        for k in keyword_counts[sector]:

            if k in text:
                matches += 1

    if matches >= 2:
        return "High"

    return "Medium"

# =========================
# RUN PIPELINE
# =========================

results = []

for idx, row in df.iterrows():

    company = row["Company Name"]

    print(f"\nProcessing: {company}")

    # -------------------------
    # SEARCH
    # -------------------------

    search_text = serper_search(company)

    # -------------------------
    # CLASSIFY
    # -------------------------

    sector = classify_sector(search_text)

    # -------------------------
    # SCORE
    # -------------------------

    c5_score = sector_score_map.get(sector, 5)

    # -------------------------
    # REASON
    # -------------------------

    reason = sector_reason_map.get(sector, "")

    # -------------------------
    # CONFIDENCE
    # -------------------------

    confidence = get_confidence(sector, search_text)

    # -------------------------
    # STORE RESULTS
    # -------------------------

    results.append({

        "Company Name": company,

        "Sector": sector,

        "C5 Score": c5_score,

        "Confidence": confidence,

        "Reason": reason,

        "Search Text": search_text[:2000]

    })

    # avoid hitting rate limits
    time.sleep(1)

# =========================
# SAVE OUTPUT
# =========================

output_df = pd.DataFrame(results)

output_df.to_excel(
    "c5_sector_scores.xlsx",
    index=False
)

print("\nDone.")
print("Saved file: c5_sector_scores.xlsx")


Processing: A R Life Sciences

Processing: lucent drugs

Processing: Hygro Chemicals

Processing: Laxai

Processing: srini chem

Processing: Sainor

Processing: Syntho Chirals

Processing: Balaji Resins and Coatings

Processing: Andhra Polymers Private Limited

Processing: Fleming Laboratories Ltd

Processing: Saptagiri Laboratories Pvt Ltd

Processing: Relixer Pharmaceuticals Pvt Ltd

Processing: Salicylates and Chemicals Pvt Ltd

Processing: Unique Biotech Ltd

Processing: Synergene Active Ingredients Pvt Ltd

Processing: Alkaloids Pvt Ltd

Processing: Valens Molecules Pvt Ltd

Processing: Veer Chemie & Aromatics Pvt Ltd

Done.
Saved file: c5_sector_scores.xlsx
